# 01 — Data workflow_update (30 mã)

**NON_BASELINE_RUN** tới khi Data Gate ký (TL-019).

Universe đã có 30 mã trên đĩa. Notebook này chạy lại từng bước `qshield-data` qua subprocess và in thời gian.

Audit: `docs/handoffs/data_30_audit.md`.


In [ ]:
import os
import subprocess
import time
from pathlib import Path


def find_root(marker="CLAUDE.md"):
    p = Path.cwd().resolve()
    for c in (p, *p.parents):
        if (c / marker).exists():
            return c
    raise RuntimeError("repo root not found")


PROJECT_ROOT = find_root()
os.chdir(PROJECT_ROOT)
CONFIG = "configs/base.yaml"
PROFILE = "configs/profiles/workflow_update.yaml"
OVERRIDE = "configs/provisional/workflow_update_downstream.yaml"
PROFILE_ARGS = ["--profile", PROFILE, "--override", OVERRIDE]
print(PROJECT_ROOT)
print("NON_BASELINE_RUN — profiled Data workflow")

In [ ]:
steps = [
    (
        "fetch",
        ["uv", "run", "qshield-data", "fetch", "--config", CONFIG, *PROFILE_ARGS],
    ),
    (
        "clean",
        ["uv", "run", "qshield-data", "clean", "--config", CONFIG, *PROFILE_ARGS],
    ),
    (
        "features",
        ["uv", "run", "qshield-data", "features", "--config", CONFIG, *PROFILE_ARGS],
    ),
    (
        "eligibility",
        ["uv", "run", "qshield-data", "eligibility", "--config", CONFIG, *PROFILE_ARGS],
    ),
    (
        "split",
        ["uv", "run", "qshield-data", "split", "--config", CONFIG, *PROFILE_ARGS],
    ),
    (
        "quality",
        ["uv", "run", "qshield-data", "quality", "--config", CONFIG, *PROFILE_ARGS],
    ),
    (
        "manifest",
        ["uv", "run", "qshield-data", "manifest", "--config", CONFIG, *PROFILE_ARGS],
    ),
]
timings = []
t0 = time.perf_counter()
for name, cmd in steps:
    print("=" * 72, flush=True)
    print(name, "$", " ".join(cmd), flush=True)
    s = time.perf_counter()
    p = subprocess.Popen(
        cmd,
        cwd=PROJECT_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert p.stdout is not None
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    elapsed = time.perf_counter() - s
    timings.append((name, elapsed, rc))
    print(f"[{name}] exit={rc} elapsed={elapsed:.1f}s", flush=True)
    if rc != 0:
        raise RuntimeError(f"{name} failed with exit {rc}")
print("TOTAL", f"{time.perf_counter() - t0:.1f}s")
for name, elapsed, rc in timings:
    print(f"{name:12s} {elapsed:8.1f}s  exit={rc}")

In [ ]:
checks = [
    "data/metadata/universe_30_asof_20260803.csv",
    "data/processed/prices_adjusted.parquet",
    "data/processed/returns.parquet",
    "data/processed/eligibility_daily.parquet",
    "reports/data_quality_report.csv",
    "reports/adjusted_close_evidence_report.csv",
    "data/metadata/data_manifest.json",
]
for rel in checks:
    p = PROJECT_ROOT / rel
    print(("OK" if p.exists() else "MISSING"), p)